In [1]:
import pandas as pd
import numpy as np

##  Load Processed Datasets

Load the Hospital Master Dataset and Bed Master Dataset from the processed data folder.

In [3]:
hospital = pd.read_csv(
    "../data/02_processed_data/hospital_master_dataset.csv",
    parse_dates=["Admission_Date", "Discharge_Date"]
)

bed = pd.read_csv(
    "../data/02_processed_data/bed_master.csv",
    parse_dates=["Admission_Date", "Discharge_Date"]
)

##  Create Hospital Final Dataset

Create a copy of the Hospital Master Dataset to prepare the final dataset for KPI engineering and Tableau dashboards.

In [5]:
hospital_final_dataset = hospital.copy()

##  Integrate Bed Information

Merge bed-related details with the Hospital Master Dataset using the common key `Admission_ID`.

In [6]:
hospital_final_dataset = hospital_final_dataset.merge(
    bed[
        [
            "Admission_ID",
            "Bed_ID",
            "Ward",
            "Floor",
            "Bed_Type",
            "Bed_Status"
        ]
    ],
    on="Admission_ID",
    how="left"
)

##  Create Additional Analytical Columns

Generate additional fields required for KPI calculation and Tableau dashboard development.

In [7]:
# Occupancy Flag
hospital_final_dataset["Occupancy_Flag"] = np.where(
    hospital_final_dataset["Bed_Status"] == "Occupied",
    1,
    0
)

# Surgery Flag
hospital_final_dataset["Had_Surgery"] = np.where(
    hospital_final_dataset["Surgery_ID"].isna(),
    "No",
    "Yes"
)

# Admission Month Number
hospital_final_dataset["Admission_Month_No"] = (
    hospital_final_dataset["Admission_Date"].dt.month
)

# Admission Day
hospital_final_dataset["Admission_Day"] = (
    hospital_final_dataset["Admission_Date"].dt.day_name()
)

##  Validate Final Dataset

Verify the structure, missing values, duplicate records, and sample data before exporting.

In [10]:
hospital_final_dataset.info()
hospital_final_dataset.head()
hospital_final_dataset.isnull().sum()
hospital_final_dataset.duplicated().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 43 columns):
 #   Column              Non-Null Count  Dtype         
---  ------              --------------  -----         
 0   Patient_ID          5000 non-null   object        
 1   Name                5000 non-null   object        
 2   Age                 5000 non-null   int64         
 3   Gender              5000 non-null   object        
 4   Diagnosis           5000 non-null   object        
 5   Age_Group           5000 non-null   object        
 6   Admission_ID        5000 non-null   object        
 7   Doctor_ID           5000 non-null   object        
 8   Department_ID       5000 non-null   object        
 9   Admission_Date      5000 non-null   datetime64[ns]
 10  Discharge_Date      5000 non-null   datetime64[ns]
 11  Status              5000 non-null   object        
 12  Length_of_Stay      5000 non-null   int64         
 13  Admission_Year      5000 non-null   int64       

np.int64(0)

##  Export Final Dataset

Save the final integrated dataset as `hospital_final_dataset.xlsx` for KPI engineering and Tableau dashboard development.

In [11]:
hospital_final_dataset.to_excel(
    "../data/02_processed_data/hospital_final_dataset.xlsx",
    index=False
)

print("Hospital Final Dataset exported successfully.")

Hospital Final Dataset exported successfully.


## Conclusion

In this notebook, the Hospital Master Dataset was enhanced with bed-related information and additional analytical fields. The final dataset was validated and exported as `hospital_final_dataset.xlsx`, making it ready for KPI engineering and Tableau dashboard development.

# Hospital KPI Engineering

## Objective

The objective of this section is to calculate the key healthcare performance indicators (KPIs) required for hospital operations analysis and Tableau dashboard development.

## 1. Load Final Dataset

Load the final hospital dataset for KPI calculation.

In [12]:
hospital_final = pd.read_excel(
    "../data/02_processed_data/hospital_final_dataset.xlsx"
)

hospital_final.head()

,Patient_ID,Name,Age,Gender,Diagnosis,Age_Group,Admission_ID,Doctor_ID,Department_ID,Admission_Date,...,Department_Name,Bed_ID,Ward,Floor,Bed_Type,Bed_Status,Occupancy_Flag,Had_Surgery,Admission_Month_No,Admission_Day
0,P00001,Patient_1,82,Female,Fever,Senior,A00001,DR00186,D002,2025-10-20,...,Neurology,B00001,Neuro Ward,4,ICU,Maintenance,0,Yes,10,Monday
1,P00002,Patient_2,31,Male,Diabetes,Young Adult,A00002,DR00281,D019,2025-01-21,...,Endocrinology,B00002,General Ward,3,ICU,Occupied,1,Yes,1,Tuesday
2,P00003,Patient_3,15,Male,Fever,Child,A00003,DR00105,D020,2025-06-24,...,Dental,B00003,General Ward,1,ICU,Reserved,0,Yes,6,Tuesday
3,P00004,Patient_4,50,Female,Asthma,Adult,A00004,DR00312,D004,2025-07-23,...,Pediatrics,B00004,Pediatric Ward,5,Private,Available,0,Yes,7,Wednesday
4,P00005,Patient_5,17,Female,Diabetes,Child,A00005,DR00430,D018,2025-04-06,...,Ophthalmology,B00005,General Ward,4,General,Occupied,1,Yes,4,Sunday


## 2. Total Admissions

Calculate the total number of patient admissions.

In [13]:
total_admissions = hospital_final["Admission_ID"].nunique()

print("Total Admissions :", total_admissions)

Total Admissions : 5000


## 3. Occupancy Rate

Calculate the percentage of occupied beds.

In [14]:
occupied_beds = (hospital_final["Occupancy_Flag"] == 1).sum()

total_beds = hospital_final["Bed_ID"].nunique()

occupancy_rate = (occupied_beds / total_beds) * 100

print("Occupancy Rate :", round(occupancy_rate,2),"%")

Occupancy Rate : 20.28 %


## 4. Average Length of Stay

Calculate the average duration of patient stay.

In [15]:
alos = hospital_final["Length_of_Stay"].mean()

print("Average Length of Stay :", round(alos,2),"Days")

Average Length of Stay : 5.52 Days


In [16]:
hospital_final.columns.tolist()

['Patient_ID',
 'Name',
 'Age',
 'Gender',
 'Diagnosis',
 'Age_Group',
 'Admission_ID',
 'Doctor_ID',
 'Department_ID',
 'Admission_Date',
 'Discharge_Date',
 'Status',
 'Length_of_Stay',
 'Admission_Year',
 'Admission_Month',
 'Admission_Quarter',
 'Weekend_Admission',
 'Stay_Category',
 'Doctor_Name',
 'Specialization',
 'Experience',
 'Bill_ID',
 'Total',
 'Insurance',
 'Paid',
 'Pending',
 'Surgery_ID',
 'Outcome',
 'Medication_ID',
 'Medicine',
 'Lab_ID',
 'Test',
 'Value',
 'Department_Name',
 'Bed_ID',
 'Ward',
 'Floor',
 'Bed_Type',
 'Bed_Status',
 'Occupancy_Flag',
 'Had_Surgery',
 'Admission_Month_No',
 'Admission_Day']

## 5.Bed Utilization Rate

Calculate the percentage of beds currently in use across the hospital.

In [17]:
occupied_beds = (hospital_final["Bed_Status"] == "Occupied").sum()

total_beds = hospital_final["Bed_ID"].nunique()

bed_utilization_rate = (occupied_beds / total_beds) * 100

print(f"Bed Utilization Rate : {bed_utilization_rate:.2f}%")

Bed Utilization Rate : 20.28%


## 6. Department-wise Admissions

Calculate the total number of admissions for each department.

In [18]:
department_admissions = (
    hospital_final.groupby("Department_Name")["Admission_ID"].count().reset_index(name="Total_Admissions"))

department_admissions

,Department_Name,Total_Admissions
0,Cardiology,233
1,Dental,253
2,Dermatology,179
3,ENT,215
4,Emergency,335
5,Endocrinology,319
6,Gastroenterology,260
7,General Surgery,175
8,Gynecology,159
9,ICU,253


## 7. Department Efficiency Score

Calculate a simple efficiency score using the ratio of patient admissions to average length of stay for each department.

In [19]:
department_efficiency = (
    hospital_final.groupby("Department_Name")
    .agg(
        Total_Admissions=("Admission_ID", "count"),
        Average_LOS=("Length_of_Stay", "mean")
    ).reset_index()
)

department_efficiency["Efficiency_Score"] = (department_efficiency["Total_Admissions"] /department_efficiency["Average_LOS"]).round(2)

department_efficiency

,Department_Name,Total_Admissions,Average_LOS,Efficiency_Score
0,Cardiology,233,5.390558,43.22
1,Dental,253,5.561265,45.49
2,Dermatology,179,5.592179,32.01
3,ENT,215,5.516279,38.98
4,Emergency,335,5.611940,59.69
5,Endocrinology,319,5.667712,56.28
6,Gastroenterology,260,5.334615,48.74
7,General Surgery,175,5.354286,32.68
8,Gynecology,159,5.427673,29.29
9,ICU,253,5.660079,44.70


In [20]:
hospital_final["Patient_ID"].duplicated().sum()

np.int64(0)

## 8. Readmission Rate

The provided dataset contains only one admission record per patient. Since there are no repeated admissions for the same patient, a true 30-day readmission rate cannot be calculated. This KPI will be documented as not applicable for the current dataset.

In [21]:
readmission_rate = None

print("Readmission Rate : Not Applicable")
print("Reason: Each Patient_ID appears only once in the dataset.")

Readmission Rate : Not Applicable
Reason: Each Patient_ID appears only once in the dataset.


## 9. Total Revenue

Calculate the total revenue generated by the hospital by summing the billing amount of all patient admissions.

This KPI helps evaluate the overall financial performance of the hospital and is useful for financial analysis and dashboard reporting.

In [22]:
total_revenue = hospital_final["Total"].sum()

print(f"Total Revenue : ₹{total_revenue:,.2f}")

Total Revenue : ₹263,393,105.00


## 10. KPI Summary

Summarize the calculated healthcare KPIs into a single table. This provides a consolidated view of the hospital's operational performance and supports Tableau dashboard development.

In [25]:
kpi_summary = pd.DataFrame({
    "KPI": [
        "Total Admissions",
        "Occupancy Rate (%)",
        "Average Length of Stay (Days)",
        "Bed Utilization Rate (%)",
        "Department Efficiency Score",
        "Total Revenue"
    ],
    "Value": [
        total_admissions,
        round(occupancy_rate, 2),
        round(alos, 2),
        round(bed_utilization_rate, 2),
        round(department_efficiency["Efficiency_Score"].mean(), 2),
        total_revenue
    ]
})

kpi_summary

,KPI,Value
0,Total Admissions,5.000000e+03
1,Occupancy Rate (%),2.028000e+01
2,Average Length of Stay (Days),5.520000e+00
3,Bed Utilization Rate (%),2.028000e+01
4,Department Efficiency Score,4.536000e+01
5,Total Revenue,2.633931e+08


In [3]:
import pandas as pd

hospital_final = pd.read_excel(
    "../data/02_processed_data/hospital_final_dataset.xlsx"
)

In [4]:
hospital_final.groupby(
    hospital_final["Admission_Date"].dt.month
)["Admission_ID"].nunique()

Admission_Date
1     516
2     479
3     504
4     508
5     519
6     505
7     524
8     506
9     500
10    439
Name: Admission_ID, dtype: int64